# Real-Time Energy Data Streaming Pipeline

This notebook implements a **real-time energy data streaming pipeline** for the Australian NEM:

1. **Data Fetching** — Pulls live power generation, emissions, and market data from the [OpenElectricity API](https://openelectricity.org.au)
2. **Data Merging** — Joins facility metadata + time-series metrics + market pricing into a unified dataset
3. **MQTT Publishing** — Streams each facility's data point to an MQTT broker for downstream consumption by the [Streamlit dashboard](../dashboard.py)

The pipeline runs in a loop, refreshing data every 60 seconds.

In [ ]:
import requests
import time
import json
from datetime import datetime
import pandas as pd
import paho.mqtt.client as mqtt
import os

while True:
    
    # Fetch facility and market data from OpenElectricity API
    
    # Configuration
    API_BASE = "https://api.openelectricity.org.au/v4"
    API_KEY = os.getenv("OPENELECTRICITY_API_KEY", "")
    HEADERS = {"Authorization": f"Bearer {API_KEY}"}
    
    # Get facility metadata
    params = {"network_id": "NEM"}
    response = requests.get(f"{API_BASE}/facilities/", headers=HEADERS, params=params)
    
    # Check status code
    if response.status_code != 200:
        print("Unable to get data:")
        print("Unable to get data:", response.text)
        exit(1)
    resp_json = response.json()
    facilities = resp_json.get("data") or resp_json.get("results") or []
    print(f"Successfully retrieved {len(facilities)} facilities.")
    
    # Save metadata
    with open("facilities_metadata.json", "w", encoding="utf-8") as f:
        json.dump(facilities, f, ensure_ascii=False, indent=2)
    print("Saved: facilities_metadata.json")
    
    # transform json to csv
    with open("facilities_metadata.json", "r", encoding="utf-8") as f:
        facilities = json.load(f)
       
    row = []
    for f in facilities:
        row.append({
            "code": f.get("code"),
            "name": f.get("name"),
            "network_region": f.get("network_region"),
            "location.lat":f.get("location", {}).get("lat"),
            "location.lng":f.get("location", {}).get("lng"),
            "units":{u.get("code"): u.get("fueltech_id") for u in f.get("units", [])}
        })
    df_facility = pd.DataFrame(row)
    df_facility["units"] = df_facility["units"].apply(json.dumps)
    df_facility.to_csv("facilities_metadata.csv", index=False)
    print("Saved: facilities_metadata.csv")
    
    # Get per-facility power generated & CO2 emissions (5 min interval)
    start ="2025-10-01T00:00:00"
    end ="2025-10-08T00:00:00"
    interval ="5m"
    table = []
    df_facility["units"] = df_facility["units"].apply(json.loads)
    for i, row in df_facility.iterrows():
        params = {
            "facility_code": row["code"],
            "metrics": ["power", "emissions"],
            "interval": interval,
            "date_start": start,
            "date_end": end
        }
        time.sleep(1)
        # fail
        response = requests.get(f"{API_BASE}/data/facilities/NEM", headers=HEADERS,params = params)
        if response.status_code != 200:
            print(f"No.{i+1}: Retrieving data for facility: {row["code"]}... Failed:", response.text)
            continue
       
        resp_json = response.json()
        
        # success
        print(f"No.{i+1}: Retrieving data for facility: {row["code"]}... Success.")
        units = row["units"]
        for f in resp_json.get("data", []):
            metric = f.get("metric")
            for u in f.get("results", []):
                unit_code = u.get("columns").get("unit_code")
                fuel_tech = units[unit_code]
                for item in u.get("data", []):
                    if isinstance(item, list) and len(item) >= 2:
                        timestamp, value = item[0], item[1]
                        table.append({"timestamp": timestamp, "facility_code": row["code"], "fuel_tech": fuel_tech, metric: value})
    
    # transform list to df
    df_metric = pd.DataFrame(table)
    if not df_metric.empty:
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'])
        df_metric = df_metric.pivot_table(index=["timestamp", "facility_code","fuel_tech"], values=["power", "emissions"], aggfunc="sum").reset_index()
        df_metric = df_metric.sort_values(by="timestamp", ascending=True).reset_index(drop=True)
    print(df_metric.shape)
    display(df_metric.head(12))
    # save csv
    df_metric.to_csv("metric.csv", index=False)
    print("Saved: metric.csv")
    
    # Get market metadata
    params = {
        "metrics": ["price", "demand"],
        "interval": interval,
        "date_start": start,
        "date_end": end
    }
    response = requests.get(f"{API_BASE}/market/network/NEM", headers=HEADERS,params = params)
    
    # Check status code
    if response.status_code != 200:
        print("Unable to get data:")
        print("Unable to get data:", response.text)
        exit(1)
    resp_json = response.json()
    market = resp_json.get("data")or resp_json.get("results")or []
    print(f"Successfully retrieved {len(market)} records.")
    
    # Save metadata
    with open("market.json", "w", encoding="utf-8") as f:
        json.dump(market, f, ensure_ascii=False, indent=2)
    print("Saved: market.json")
    
    # transform json to csv
    market_table = []
    for m in market:
        metric = m.get("metric")
        for u in m.get("results", []):
            for item in u.get("data", []):
                if isinstance(item, list) and len(item) >= 2:
                    timestamp, value = item[0], item[1]
                    market_table.append({"timestamp": timestamp, metric: value})
                   
    df_market = pd.DataFrame(market_table)
    df_market = df_market.pivot_table(index="timestamp", values=["price", "demand"], aggfunc="first").reset_index()
    df_market['timestamp'] = pd.to_datetime(df_market['timestamp'])
    df_market = df_market.sort_values(by="timestamp").reset_index(drop=True)
    df_market.to_csv("market_metadata.csv", index=False)
    print("Saved: market_metadata.csv")


    
    # Merge facility metadata, metrics, and market data
    # merge
    df_facility = pd.read_csv("facilities_metadata.csv")
    df_metric = pd.read_csv("metric.csv")
    df_market = pd.read_csv("market_metadata.csv")
    df_total = pd.merge(df_metric, df_facility, left_on="facility_code", right_on="code", how="left")
    df_total = pd.merge(df_total, df_market, left_on="timestamp", right_on="timestamp", how="left")
    print(df_total.shape)
    display(df_total.head(12))
    # missing value
    df_total = df_total.dropna(subset=["power", "emissions", "demand", "price"])
    # negative value
    df_total = df_total[(df_total["power"] >= 0) & (df_total["emissions"] >= 0) & (df_total["demand"] >= 0)]
    # delete irrelevant column
    df_total = df_total[['timestamp', 'name', 'fuel_tech', 'power', 'emissions', 'network_region', 'price', 'demand', 'location.lat', 'location.lng']]
    print(df_total.shape)
    df_total.to_csv("total.csv", index=False)
    print(f"Saved: total.csv")
    display(df_total.head(12))


    
    # Publish merged data via MQTT
    df = pd.read_csv("total.csv")
    def on_connect(client, userdata, connect_flags, reason_code, properties):
        print("Connected with result code " + str(reason_code))
        client.subscribe("NEM/PowerEmissions")
    # Define what happens upon receiving a message from the server
    def on_message(client, userdata, msg):
        print(f"Received message on topic {msg.topic}: {msg.payload.decode()}")
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    # Connect to the server using the IP address and connection port
    client.connect("test.mosquitto.org", 1883, 60)
    client.loop_start()
   
    
    #client.publish(topic, payload="", retain=True)
    topic = "NEM/PowerEmissions"
    prev_pair = {}
    for i, row in df.iterrows():
        send_message = True
        facility_name = row["name"]
        fuel_tech = row["fuel_tech"]
        con = facility_name + fuel_tech
        if con in prev_pair:
            if prev_pair[con][0] == 0 and prev_pair[con][1] == 0:
                if row["power"] == 0 and row["emissions"] == 0:
                    send_message = False
        if send_message:
            package = {
                "timestamp": row["timestamp"],
                "name": facility_name,
                "fuel_tech": fuel_tech,
                "power": row["power"],
                "emissions": row["emissions"],
                'network_region': row["network_region"],
                "price": row["price"],
                "demand": row["demand"],
                'lat':row["location.lat"],
                'lon':row["location.lng"]
            }
            client.publish(topic, json.dumps(package),qos=1, retain=True)
            print(f"Sent No.{i+1}: {package}")
        prev_pair[con] = [row["power"], row["emissions"]]
        time.sleep(0.1)
    client.loop_stop()

    # Additional delay between API retrievals
    time.sleep(60)

### Pipeline Details

**Fetching —** Gets all NEM facilities from `/facilities/`, retrieves 5-minute interval power & emissions for each facility via `/data/facilities/NEM`, and fetches market-wide price & demand from `/market/network/NEM`. Rate-limited to 1 request/sec.

**Merging —** Left-joins facility metadata + metrics on `facility_code`, then left-joins market data on `timestamp`. Drops rows with missing values or negative readings. Outputs `total.csv`.

**MQTT Streaming —** Reads `total.csv` row by row, publishes each as JSON to `NEM/PowerEmissions`. Skips consecutive zero-power + zero-emissions rows to reduce noise. QoS 1 with retain flag for reliable delivery.